# NexusAI Core-01 — FineWeb 100M Corpus + First Pretraining Workflow

**Status: PREPARATION NOTEBOOK — nothing runs automatically.**

This notebook walks Core-01 from UNTRAINED to its first real pretraining run:
FineWeb shard download → 100M-token corpus prep → dataset audit → GPU smoke test → full training.

| Stage | Accelerator | Owner approval required? |
|---|---|---|
| Env/package/tokenizer checks | CPU | No |
| FineWeb shard download (~2.0 GiB) | CPU | **YES — `APPROVE_FINEWEB_DOWNLOAD`** |
| Corpus preparation (100M tokens) | CPU | **YES — `APPROVE_CORPUS_PREPARATION`** |
| Dataset audit | CPU | No |
| GPU smoke test (real data, ~5–10M tok) | **GPU T4** | **YES — `APPROVE_GPU_SMOKE_TEST`** |
| Full 100M pretraining | **GPU T4** | **YES — `APPROVE_FULL_TRAINING`** |

- Raw data → `/kaggle/working/nexusai_core01/raw/`
- Prepared corpus → `/kaggle/working/nexusai_core01/prepared_dataset/`
- Checkpoints → `/kaggle/working/nexusai_core01/checkpoints/`
- Package extraction → `/kaggle/working/nexusai_core01/pkg/`

**How to stop safely:** every gated stage defaults to `False`; set a flag to `True` only after reading the printed plan. `KeyboardInterrupt` during training saves `checkpoint_interrupted.pt` (status stays `PRETRAINING_IN_PROGRESS`).

## Cell 2 — Environment Inspection (read-only, no installs)

In [ ]:
import os, sys, platform, shutil, subprocess

print("python:", sys.version.split()[0], "|", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| cuda_build:", torch.version.cuda,
          "| cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("vram_total_GB: %.1f" % (torch.cuda.get_device_properties(0).total_memory/1e9))
except Exception as e:
    print("torch: NOT AVAILABLE ->", e)
try:
    import sentencepiece; print("sentencepiece:", sentencepiece.__version__)
except ImportError:
    print("sentencepiece: MISSING -> Settings > Install packages: sentencepiece")
try:
    import pyarrow; print("pyarrow:", pyarrow.__version__)
except ImportError:
    print("pyarrow: MISSING -> needed only for corpus prep; Settings > Add package: pyarrow")
try:
    out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10)
    print("nvidia-smi:", out.stdout.strip() or "(none)")
except Exception as e:
    print("nvidia-smi: unavailable ->", e)
for p in ["/kaggle/working", "/kaggle/input", "."]:
    try:
        u = shutil.disk_usage(p); print(f"disk {p}: free {u.free/1e9:.1f} GB")
    except Exception: pass
print("cpu_count:", os.cpu_count())
try:
    import psutil; print("ram_GB: %.1f" % (psutil.virtual_memory().total/1e9))
except Exception:
    print("ram: psutil missing (optional)")

## Cell 3 — Kaggle Input / Package Detection

Detects `nexusai_core01_kaggle_package.zip` (or an extracted folder) under `/kaggle/input`,
extracts to `/kaggle/working/nexusai_core01/pkg/`, verifies SHA-256, and locates
`core01/`, `models/`, tokenizer, and `data/prepare_english_corpus.py`.
`/kaggle/input` is never modified. `PACKAGE_OK=False` blocks all later cells.

In [ ]:
import os, sys, glob, zipfile, hashlib, shutil

PACKAGE_OK = False
PACKAGE_ROOT = None      # dir containing core01/ and models/
PKG_DIR = "/kaggle/working/nexusai_core01/pkg"
TOKENIZER_PATH = None
PREP_SCRIPT = None
EXPECTED_SHA = "1b0c66f03457eb764b08dacde5594e4093581b2b3f116ad9e3176a97d33be970"

zip_candidates = sorted(glob.glob("/kaggle/input/**/nexusai_core01_kaggle_package*.zip", recursive=True))
dir_candidates = sorted(glob.glob("/kaggle/input/**/nexusai_core01_kaggle_package", recursive=True))
print("zip candidates:", zip_candidates)
print("dir candidates:", dir_candidates)

try:
    if zip_candidates:
        zpath = zip_candidates[0]
        h = hashlib.sha256()
        with open(zpath, "rb") as f:
            for chunk in iter(lambda: f.read(1<<20), b""): h.update(chunk)
        print(f"zip sha256: {h.hexdigest()}")
        print("sha256 match:", h.hexdigest() == EXPECTED_SHA)
        if os.path.exists(PKG_DIR): shutil.rmtree(PKG_DIR)
        os.makedirs(PKG_DIR, exist_ok=True)
        with zipfile.ZipFile(zpath) as z: z.extractall(PKG_DIR)
    elif dir_candidates:
        print("using extracted package dir (sha check skipped)")
        if os.path.exists(PKG_DIR): shutil.rmtree(PKG_DIR)
        shutil.copytree(dir_candidates[0], PKG_DIR)
    else:
        raise FileNotFoundError("Package not found: attach nexusai_core01_kaggle_package.zip as Kaggle Input.")

    # locate the real package root (contains core01/ and models/)
    roots = [d for d in glob.glob(PKG_DIR + "/**", recursive=True)
             if os.path.isdir(os.path.join(d, "core01")) and os.path.isdir(os.path.join(d, "models"))]
    if not roots: raise FileNotFoundError("core01/+models/ not found inside package")
    PACKAGE_ROOT = roots[0]
    print("PACKAGE_ROOT:", PACKAGE_ROOT)

    for req in ["core01/__init__.py", "core01/dataset.py", "core01/training.py",
                "core01/pretraining_driver.py", "core01/transformer.py",
                "models/nexusai_core_01/config/model_config.py"]:
        assert os.path.exists(os.path.join(PACKAGE_ROOT, req)), f"missing {req}"

    toks = glob.glob(PACKAGE_ROOT + "/**/sp_bpe_8k.model", recursive=True)
    assert toks, "sp_bpe_8k.model not found"
    TOKENIZER_PATH = toks[0]
    print("tokenizer:", TOKENIZER_PATH)

    preps = glob.glob(PACKAGE_ROOT + "/**/prepare_english_corpus.py", recursive=True)
    PREP_SCRIPT = preps[0] if preps else None
    print("prep script:", PREP_SCRIPT)

    if PACKAGE_ROOT not in sys.path: sys.path.insert(0, PACKAGE_ROOT)
    if os.path.join(PACKAGE_ROOT, "models") not in sys.path:
        sys.path.insert(0, os.path.join(PACKAGE_ROOT, "models"))
    PACKAGE_OK = True
    print("PACKAGE_OK =", PACKAGE_OK)
except Exception as e:
    PACKAGE_OK = False
    print("PACKAGE INIT FAILED:", repr(e))
    print("Fix: Kaggle -> your notebook -> Add Input -> select the nexusai_core01_kaggle_package dataset, then re-run this cell.")

## Cell 4 — Core-01 Tokenizer Verification (read-only)

Verifies vocab=8000, special IDs (pad=0, unk=1, bos=2, eos=3), file SHA-256,
and a real encode/decode round-trip. Never substitutes another tokenizer.

In [ ]:
assert PACKAGE_OK, "Package init failed — fix Cell 3 first."
import hashlib, sys, os
sys.path.insert(0, PACKAGE_ROOT)
sys.path.insert(0, os.path.join(PACKAGE_ROOT, "models"))

from core01.tokenizer_interface import NexusAITokenizer
from nexusai_core_01.config.model_config import NexusAICore01Config

cfg = NexusAICore01Config()
tok = NexusAITokenizer(TOKENIZER_PATH, cfg)
vocab = tok.get_vocab_size()
print("vocab_size:", vocab, "(expected 8000)")
print("special IDs: pad=%d unk=%d bos=%d eos=%d" % (tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id))
assert vocab == 8000, "vocab mismatch"
assert (tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id) == (0,1,2,3), "special ID mismatch"

h = hashlib.sha256()
with open(TOKENIZER_PATH, "rb") as f:
    for c in iter(lambda: f.read(1<<20), b""): h.update(c)
print("tokenizer sha256:", h.hexdigest())

ids = tok.encode("Hello, this is a tokenizer verification test.", add_special_tokens=True)
print("encode sample ids:", ids[:15], "... len", len(ids))
back = tok.decode(ids)
print("decode round-trip ok:", isinstance(back, str) and len(back) > 0)
TOKENIZER_OK = True

## Cell 5 — FineWeb Source Verification (read-only metadata only)

Checks the official HF repo, `sample-10BT` config, shard-0 URL, content length,
HTTP Range support, and license metadata. **Makes NO download.**
License: ODC-By 1.0 (attribution required) + Common Crawl Terms of Use.
Underlying page copyrights remain with their authors — this corpus is NOT
license-free; owner acknowledges this before approving download.

In [ ]:
import json, urllib.request

FINEWEB_API = "https://huggingface.co/api/datasets/HuggingFaceFW/fineweb/parquet/sample-10BT"
SHARD_URL = FINEWEB_API + "/train/0.parquet"

try:
    meta = json.load(urllib.request.urlopen(
        "https://huggingface.co/api/datasets/HuggingFaceFW/fineweb", timeout=30))
    cd = meta.get("cardData") or {}
    print("repo: HuggingFaceFW/fineweb | config: sample-10BT")
    print("license:", cd.get("license"), "| language:", cd.get("language"))
    print("attribution required: ODC-By 1.0 — cite 'FineWeb by Hugging Face'")
    print("Common Crawl ToS: https://commoncrawl.org/terms-of-use")

    shards = json.load(urllib.request.urlopen(FINEWEB_API, timeout=30))["train"]
    print("sample-10BT shards:", len(shards))

    req = urllib.request.Request(SHARD_URL, method="HEAD")
    r = urllib.request.urlopen(req, timeout=30)
    size = int(r.headers.get("Content-Length", 0))
    print("shard0 URL resolves:", r.status == 200)
    print("shard0 size bytes:", size, "= %.2f GiB" % (size/2**30))

    # Range support probe (1 byte, no real download)
    try:
        rr = urllib.request.Request(SHARD_URL, headers={"Range":"bytes=0-0"})
        r2 = urllib.request.urlopen(rr, timeout=30)
        print("HTTP Range support:", r2.status in (200,206))
    except Exception as e:
        print("Range probe failed (non-fatal):", e)

    FINEWEB_VERIFIED = (size > 1_500_000_000)
    print("FINEWEB_VERIFIED:", FINEWEB_VERIFIED)
except Exception as e:
    FINEWEB_VERIFIED = False
    print("FineWeb verification FAILED:", repr(e))
    print("Check internet access in Kaggle (Settings > Internet ON) and retry.")

## Cell 6 — FineWeb Shard Download — **OWNER APPROVAL GATE**

`APPROVE_FINEWEB_DOWNLOAD` defaults to **False**. While False this cell does
NOT touch the network. Set True only after Cell 5 verification and your
explicit approval. Downloads ONLY shard 0 with streaming + progress,
verifies size + SHA-256, never fetches more shards.

In [ ]:
APPROVE_FINEWEB_DOWNLOAD = False   # <-- owner sets True to authorize

RAW_DIR = "/kaggle/working/nexusai_core01/raw"
SHARD_PATH = os.path.join(RAW_DIR, "fineweb_sample10bt_shard0.parquet")
DOWNLOAD_OK = False

if not APPROVE_FINEWEB_DOWNLOAD:
    print("DOWNLOAD BLOCKED: APPROVE_FINEWEB_DOWNLOAD=False (default).")
    print("To authorize: set APPROVE_FINEWEB_DOWNLOAD = True after reviewing Cell 5.")
else:
    import urllib.request, time
    os.makedirs(RAW_DIR, exist_ok=True)
    # re-verify size + disk before downloading
    head = urllib.request.urlopen(urllib.request.Request(SHARD_URL, method="HEAD"), timeout=30)
    expected = int(head.headers.get("Content-Length", 0))
    free = shutil.disk_usage(RAW_DIR).free
    print(f"expected={expected/1e9:.2f} GB  disk_free={free/1e9:.1f} GB")
    assert free > expected * 1.2, "Insufficient disk for shard + margin"

    if os.path.exists(SHARD_PATH) and os.path.getsize(SHARD_PATH) == expected:
        print("shard already present with correct size; skipping download")
    else:
        t0 = time.time()
        req = urllib.request.Request(SHARD_URL)
        with urllib.request.urlopen(req, timeout=60) as r, \
             open(SHARD_PATH + ".part", "wb") as f:
            total = int(r.headers.get("Content-Length", expected))
            got = 0; last = 0
            while True:
                chunk = r.read(1 << 22)
                if not chunk: break
                f.write(chunk); got += len(chunk)
                if time.time() - last > 5:
                    print(f"  {got/1e9:.2f}/{total/1e9:.2f} GB "
                          f"({100*got/total:.0f}%) {got/(time.time()-t0)/1e6:.1f} MB/s", flush=True)
                    last = time.time()
        os.replace(SHARD_PATH + ".part", SHARD_PATH)
        print(f"download done in {time.time()-t0:.0f}s")

    actual = os.path.getsize(SHARD_PATH)
    print("size check:", actual == expected, f"({actual} bytes)")
    h = hashlib.sha256()
    with open(SHARD_PATH, "rb") as f:
        for c in iter(lambda: f.read(1<<22), b""): h.update(c)
    print("shard sha256:", h.hexdigest())
    DOWNLOAD_OK = (actual == expected)
    print("DOWNLOAD_OK:", DOWNLOAD_OK)

## Cell 7 — 100M Token Corpus Preparation — **OWNER APPROVAL GATE**

Runs the EXISTING `data/prepare_english_corpus.py` from the package (not a copy).
Target 100,000,000 Core-01 SentencePiece tokens, 0.5% buffer, 1.5% val, seed 42.
CPU-only. `APPROVE_CORPUS_PREPARATION` defaults to False.

In [ ]:
APPROVE_CORPUS_PREPARATION = False   # <-- owner sets True to authorize

PREPARED_DIR = "/kaggle/working/nexusai_core01/prepared_dataset"
PREP_DONE = False

if not APPROVE_CORPUS_PREPARATION:
    print("PREP BLOCKED: APPROVE_CORPUS_PREPARATION=False (default).")
else:
    assert PACKAGE_OK and PREP_SCRIPT, "package/prep script missing"
    assert DOWNLOAD_OK or os.path.exists(SHARD_PATH), "shard not downloaded"
    cmd = [sys.executable, PREP_SCRIPT,
           "--input-parquet", SHARD_PATH,
           "--tokenizer", TOKENIZER_PATH,
           "--output-dir", PREPARED_DIR,
           "--target-tokens", "100000000",
           "--buffer-fraction", "0.005",
           "--val-fraction", "0.015",
           "--seed", "42",
           "--source-name", "fineweb",
           "--source-config", "sample-10BT",
           "--source-license", "ODC-By-1.0"]
    print("running:", " ".join(cmd))
    import subprocess
    r = subprocess.run(cmd)
    print("prep exit code:", r.returncode)
    PREP_DONE = (r.returncode == 0)
    for f in ["english_train.jsonl","english_val.jsonl","manifest.json",
              "preparation_report.md","deduplication_report.json"]:
        p = os.path.join(PREPARED_DIR, f)
        print(("OK " if os.path.exists(p) else "MISSING ") + f,
              os.path.getsize(p) if os.path.exists(p) else "")

## Cell 8 — Dataset Verification (read-only)

Re-counts tokens with the real tokenizer, checks splits, duplicates,
manifest hashes, and driver compatibility. Prints actuals — never estimates.

In [ ]:
DATASET_OK = False
train_path = os.path.join(PREPARED_DIR, "english_train.jsonl")
val_path = os.path.join(PREPARED_DIR, "english_val.jsonl")
man_path = os.path.join(PREPARED_DIR, "manifest.json")

if not (os.path.exists(train_path) and os.path.exists(val_path)):
    print("No prepared dataset found — run Cell 7 first (requires approval).")
else:
    import json
    tr = [json.loads(l) for l in open(train_path, encoding="utf-8") if l.strip()]
    va = [json.loads(l) for l in open(val_path, encoding="utf-8") if l.strip()]
    tr_tok = sum(d["ntok"] for d in tr); va_tok = sum(d["ntok"] for d in va)
    print(f"train: {len(tr):,} docs / {tr_tok:,} tokens")
    print(f"val:   {len(va):,} docs / {va_tok:,} tokens")
    tr_texts = {d["text"] for d in tr}; va_texts = {d["text"] for d in va}
    print("overlap docs:", len(tr_texts & va_texts))
    print("empty docs:", sum(1 for d in tr+va if not d["text"].strip()))
    print("dup docs in outputs:", (len(tr)+len(va)) - len(tr_texts|va_texts))
    man = json.load(open(man_path, encoding="utf-8"))
    print("manifest actual_tokens:", man["target"]["actual_accepted_tokens"])
    print("manifest train/val:", man["split"]["train_tokens"], "/", man["split"]["val_tokens"])
    # driver compatibility: load_texts
    sys.path.insert(0, PACKAGE_ROOT)
    from core01.pretraining_driver import load_texts
    docs = load_texts(train_path)
    print("driver load_texts:", len(docs), "docs OK")
    DATASET_OK = (len(tr_texts & va_texts) == 0 and tr_tok + va_tok > 0)
    print("DATASET_OK:", DATASET_OK)

## Cell 9 — GPU Environment Verification (read-only)

Checks cuda:0 only. If unavailable, enable GPU in Kaggle: Notebook → Settings →
Accelerator → GPU T4 x2 (we use ONE GPU). No forward/backward/training here.

In [ ]:
GPU_OK = False
import torch
if not torch.cuda.is_available():
    print("CUDA unavailable -> Notebook Settings > Accelerator > GPU T4 x2, then re-run.")
else:
    props = torch.cuda.get_device_properties(0)
    print("device 0:", props.name, "| vram %.1f GB" % (props.total_memory/1e9),
          "| capability", props.major, props.minor)
    print("bf16_supported:", torch.cuda.is_bf16_supported(),
          "(T4=sm_75 -> expect False; fp16 path will be used)")
    GPU_OK = True
print("GPU_OK:", GPU_OK)

## Cell 10 — Real-Data GPU Smoke Test — **OWNER APPROVAL GATE**

Prepared only. Validates the REAL pipeline (PackedTextDataset + driver +
PretrainingEngine + FP16 AMP + GradScaler + checkpoints) on actual corpus
subset (~5–10M tokens max, small step count). Single GPU cuda:0.
`APPROVE_GPU_SMOKE_TEST` defaults False. Requires Cells 3,4,7,8,9 OK.

In [ ]:
APPROVE_GPU_SMOKE_TEST = False   # <-- owner sets True after reviewing plan

SMOKE_DONE = False
if not APPROVE_GPU_SMOKE_TEST:
    print("SMOKE TEST BLOCKED: APPROVE_GPU_SMOKE_TEST=False (default).")
else:
    assert PACKAGE_OK and TOKENIZER_OK and DATASET_OK and GPU_OK,         "preconditions failed (package/tokenizer/dataset/gpu)"
    CKPT_DIR = "/kaggle/working/nexusai_core01/checkpoints_smoke"
    # Pilot: bounded real-data run — adjust --max-steps after seeing step time.
    cmd = [sys.executable, os.path.join(PACKAGE_ROOT, "core01", "pretraining_driver.py"),
           "--train-file", train_path,
           "--val-file", val_path,
           "--output-dir", CKPT_DIR,
           "--tokenizer", TOKENIZER_PATH,
           "--device", "cuda",
           "--precision", "fp16",
           "--batch-size", "32",
           "--gradient-accumulation-steps", "1",
           "--learning-rate", "3e-4",
           "--warmup-steps", "100",
           "--epochs", "1",
           "--max-steps", "200",
           "--checkpoint-interval", "100",
           "--eval-interval", "100",
           "--log-interval", "10"]
    print("SMOKE cmd:", " ".join(cmd))
    import subprocess
    r = subprocess.run(cmd)
    SMOKE_DONE = (r.returncode == 0)
    print("SMOKE_DONE:", SMOKE_DONE,
          "| review loss curve + val_loss + VRAM before approving full run")

## Cell 11 — Full 100M Pretraining — **OWNER APPROVAL GATE**

Prepared only. Final config MUST come from Cell 10 measurements
(step time, tokens/sec, VRAM, checkpoint size) — do not guess.
`APPROVE_FULL_TRAINING` defaults False. Single T4 (`cuda:0`), FP16 AMP.
Resume-safe: checkpoints every N steps; Kaggle session limits handled by
`--resume checkpoint_last.pt` in a new session if needed.

In [ ]:
APPROVE_FULL_TRAINING = False   # <-- owner sets True ONLY after pilot review

FULL_DONE = False
if not APPROVE_FULL_TRAINING:
    print("FULL TRAINING BLOCKED: APPROVE_FULL_TRAINING=False (default).")
    print("Required before enabling: SMOKE_DONE=True and owner sign-off on")
    print("batch size / LR / step budget derived from measured pilot throughput.")
else:
    assert SMOKE_DONE, "run and verify the GPU smoke test first"
    CKPT_DIR = "/kaggle/working/nexusai_core01/checkpoints_full"
    # PLACEHOLDER config — replace values with pilot-measured numbers.
    cmd = [sys.executable, os.path.join(PACKAGE_ROOT, "core01", "pretraining_driver.py"),
           "--train-file", train_path,
           "--val-file", val_path,
           "--output-dir", CKPT_DIR,
           "--tokenizer", TOKENIZER_PATH,
           "--device", "cuda",
           "--precision", "fp16",
           "--batch-size", "32",          # TODO: set from pilot VRAM/step-time
           "--gradient-accumulation-steps", "1",
           "--learning-rate", "3e-4",     # TODO: confirm from pilot loss curve
           "--warmup-steps", "500",
           "--epochs", "3",               # ~3 passes over ~98.5M tokens
           "--max-steps", "0",            # 0 = run all epochs
           "--checkpoint-interval", "500",
           "--eval-interval", "500",
           "--log-interval", "50"]
    print("FULL cmd:", " ".join(cmd))
    print("Estimated steps: ~%d optimizer steps (placeholder — compute from pilot tok/s)"
          % (98500000 // (32*512)))
    import subprocess
    r = subprocess.run(cmd)
    FULL_DONE = (r.returncode == 0)
    print("FULL_DONE:", FULL_DONE,
          "| verify checkpoint_final.pt status=PRETRAINING_COMPLETED")

## Cell 12 — Final Status Report

Run this cell anytime to see the current stage status.

In [ ]:
def _st(cond, yes="COMPLETED", no="NOT STARTED"):
    return yes if cond else no

print("=" * 55)
print("NEXUSAI CORE-01 PIPELINE STATUS")
print("=" * 55)
print("environment        :", "VERIFIED" if 'torch' in globals() else "NOT STARTED")
print("package            :", _st(globals().get("PACKAGE_OK"), "VERIFIED", "BLOCKED"))
print("tokenizer          :", _st(globals().get("TOKENIZER_OK"), "VERIFIED", "NOT STARTED"))
print("fineweb verify     :", _st(globals().get("FINEWEB_VERIFIED"), "VERIFIED", "NOT STARTED"))
print("download           :", _st(globals().get("DOWNLOAD_OK"), "COMPLETED", "BLOCKED"))
print("corpus prep        :", _st(globals().get("PREP_DONE"), "COMPLETED", "BLOCKED"))
print("dataset audit      :", _st(globals().get("DATASET_OK"), "VERIFIED", "NOT STARTED"))
print("gpu env            :", _st(globals().get("GPU_OK"), "VERIFIED", "NOT STARTED"))
print("gpu smoke test     :", _st(globals().get("SMOKE_DONE"), "COMPLETED", "BLOCKED"))
print("full training      :", _st(globals().get("FULL_DONE"), "COMPLETED", "BLOCKED"))
print("=" * 55)
print("CORE-01 STATUS: UNTRAINED (until checkpoint_final.pt verified)")